# Removing Redundant Columns Using Correlation Coefficient

## Feature Redundancy in Datasets

In many real-world datasets, some features may contain similar or overlapping information. When two variables are highly related, they are considered **redundant**. Keeping both may not add significant value and can negatively impact model performance.

Removing redundant columns is a form of **feature selection**, which helps in simplifying the dataset while preserving essential information.


## Correlation Coefficient

The correlation coefficient measures the strength and direction of the linear relationship between two variables.

### Pearson Correlation Coefficient Formula

$$
r = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y}
$$

Where:

* $\text{Cov}(X, Y)$ = covariance between X and Y
* $\sigma_X$ = standard deviation of variable X
* $\sigma_Y$ = standard deviation of variable Y

The correlation coefficient standardizes covariance, making the value scale-independent.

## Covariance Formula

$$
\text{Cov}(X, Y) = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{n - 1}
$$

Where:

* $\bar{x}$ = mean of X
* $\bar{y}$ = mean of Y
* $n$ = number of observations


## Interpretation of Correlation Coefficient

The value of correlation ranges between **-1 and +1**.

* **+1** → Perfect positive linear relationship
* **-1** → Perfect negative linear relationship
* **0** → No linear relationship

### Strength of Correlation

* 0.0 – 0.3 → Weak relationship
* 0.3 – 0.7 → Moderate relationship
* 0.7 – 0.9 → Strong relationship
* Above 0.9 → Very strong relationship (high redundancy risk)

## Why Remove Highly Correlated Columns?

When two features are highly correlated:

* They provide nearly identical information
* They introduce multicollinearity
* They increase model complexity
* They may destabilize regression coefficients
* They reduce interpretability of the model

Removing one of the correlated features helps in building a more stable and efficient model.


## How to Decide Which Column to Remove

When correlation is very high:

* Remove the feature that has less predictive importance
* Remove the feature with more missing values
* Remove the feature that has lower domain relevance
* Keep the feature that has stronger relationship with the target variable

Feature selection should always consider both statistical results and domain knowledge.


## Benefits of Removing Redundant Columns

* Reduces dimensionality
* Improves computational efficiency
* Prevents overfitting
* Enhances model stability
* Improves interpretability
* Reduces noise in the dataset



The notebook begins by importing the essential Python libraries used for data analysis and visualization.

* **NumPy** is used for numerical computations and working with arrays and mathematical operations.
* **Pandas** is used for data manipulation and handling structured datasets such as tables (DataFrames).
* **Matplotlib** is a visualization library used for creating basic plots and graphs.
* **Seaborn** is built on top of Matplotlib and provides advanced statistical visualizations with better styling.

These libraries form the foundation for performing data preprocessing, analysis, and visualization tasks in the notebook.


In [1]:
# Import Libaries required
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Reading Data from CSV

In this step, the dataset is loaded into the notebook using Pandas. The `read_csv` function reads the CSV file and stores it in a DataFrame named `df`. Using `df.head()` displays the first few records, allowing an initial look at the structure and columns of the data before further analysis.

The dataset used is the **Heart Disease Dataset**, which can be found here:
[Click here](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset)


In [7]:
# Reading Data From CSV
df = pd.read_csv(r'C:\Users\athar\Downloads\dataset-1.csv')
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


## Dataset Cleaning and Data Quality Verification

### Handling Missing Values

After identifying missing values, appropriate strategies must be applied depending on the nature of the data.

#### 1. Replacing with Mean

Missing values in numerical columns can be replaced with the **mean** of that column.

* Suitable when data is normally distributed
* Preserves overall average
* Sensitive to outliers


```python
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())
```


Mean replacement is commonly used for continuous numerical variables.

#### 2. Replacing with Median

Missing values can be replaced with the **median** of the column.

* More robust to outliers
* Suitable for skewed distributions
* Maintains central tendency without distortion

```python
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())
```

Median replacement is preferred when extreme values are present.

#### 3. Replacing with Mode

For categorical variables or discrete numerical variables, missing values can be replaced with the **mode** (most frequent value).

* Maintains the most common category
* Useful for classification problems


```python
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mode())
```


### Dropping Duplicate Rows

Duplicate entries are removed to maintain dataset integrity.

Duplicate records may:

* Inflate dataset size
* Bias statistical results
* Negatively affect model training

Removing duplicates ensures each observation is unique and meaningful.


In [18]:
# Check dataset shape before cleaning
rows, columns = df.shape
print(f"Dataset contains {rows} entries and {columns} attributes BEFORE cleaning.")

# Check missing values
print("Missing Values Before Handling:")
print(df.isnull().sum())

# Drop duplicate rows
df = df.drop_duplicates()

# Check dataset shape after cleaning
rows, columns = df.shape
print(f"Dataset contains {rows} entries and {columns} attributes AFTER cleaning.")

Dataset contains 302 entries and 14 attributes BEFORE cleaning.
Missing Values Before Handling:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64
Dataset contains 302 entries and 14 attributes AFTER cleaning.


## Outlier Detection and Capping Using IQR Method

### Selecting Numerical Columns

Only numerical columns are selected for Interquartile Range (IQR) calculation because:

* Quartile computation is meaningful only for numerical data
* Categorical variables cannot be processed using statistical dispersion measures

This ensures that outlier detection is applied appropriately.

### Outlier Capping (Winsorization)

Instead of removing outliers, extreme values are capped using the calculated lower and upper bounds.

This process:

* Replaces values below the lower bound with the lower bound
* Replaces values above the upper bound with the upper bound

Advantages of capping:

* Preserves dataset size
* Reduces influence of extreme values
* Maintains overall data structure
* Improves statistical stability

### Result

After applying capping:

* Extreme values are controlled
* Data distribution becomes more stable
* Central tendency measures become more reliable
* Dataset is better prepared for scaling and modeling

This completes the outlier treatment stage of data preprocessing.


In [19]:
# Select only numerical columns for IQR calculation
num_df = df.select_dtypes(include='number')

# Calculate first quartile (Q1) and third quartile (Q3)
q1 = num_df.quantile(0.25)
q3 = num_df.quantile(0.75)

# Calculate Inter-Quartile Range (IQR)
iqr = q3 - q1
print("Inter-Quartile Range:\n", iqr)

# Calculate lower and upper bounds
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("\nLower Bound:\n", lower_bound)
print("\nUpper Bound:\n", upper_bound)

# Cap outliers using clip (Winsorization)
new_df = df.copy()
new_df[num_df.columns] = num_df.clip(lower=lower_bound, upper=upper_bound, axis=1)

print("\nOutlier capping completed.")

Inter-Quartile Range:
 age         13.00
sex          1.00
cp           2.00
trestbps    20.00
chol        63.75
fbs          0.00
restecg      1.00
thalach     32.75
exang        1.00
oldpeak      1.60
slope        1.00
ca           1.00
thal         1.00
target       1.00
dtype: float64

Lower Bound:
 age          28.500
sex          -1.500
cp           -3.000
trestbps     90.000
chol        115.375
fbs           0.000
restecg      -1.500
thalach      84.125
exang        -1.500
oldpeak      -2.400
slope        -0.500
ca           -1.500
thal          0.500
target       -1.500
dtype: float64

Upper Bound:
 age          80.500
sex           2.500
cp            5.000
trestbps    170.000
chol        370.375
fbs           0.000
restecg       2.500
thalach     215.125
exang         2.500
oldpeak       4.000
slope         3.500
ca            2.500
thal          4.500
target        2.500
dtype: float64

Outlier capping completed.


In [14]:
# Dataset After Capping the Outliers
new_df.head(10)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,302.00000,302.000000,302.000000,302.000000,302.000000,302.0,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000
mean,54.42053,0.682119,0.963576,131.258278,245.377070,0.0,0.526490,149.612997,0.327815,1.027815,1.397351,0.665563,2.317881,0.543046
std,9.04797,0.466426,1.032044,16.605232,47.486683,0.0,0.526027,22.765983,0.470196,1.110395,0.616274,0.880241,0.601724,0.498970
min,29.00000,0.000000,0.000000,94.000000,126.000000,0.0,0.000000,84.125000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000
25%,48.00000,0.000000,0.000000,120.000000,211.000000,0.0,0.000000,133.250000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000
50%,55.50000,1.000000,1.000000,130.000000,240.500000,0.0,1.000000,152.500000,0.000000,0.800000,1.000000,0.000000,2.000000,1.000000
75%,61.00000,1.000000,2.000000,140.000000,274.750000,0.0,1.000000,166.000000,1.000000,1.600000,2.000000,1.000000,3.000000,1.000000
max,77.00000,1.000000,3.000000,170.000000,370.375000,0.0,2.000000,202.000000,1.000000,4.000000,2.000000,2.500000,3.000000,1.000000


## Manual Statistical Calculations After Outlier Capping

After performing outlier capping, statistical measures are calculated manually without using built-in statistical functions. This ensures a clear understanding of the mathematical foundation behind each concept.



## Central Tendency Measures

### Mean

The mean represents the average value of the dataset.

$$
\bar{x} = \frac{\sum x_i}{n}
$$

Where:

* $x_i$ = individual observations
* $n$ = total number of observations

The mean is sensitive to extreme values, which is why outlier capping is performed beforehand.


### Median

The median is the middle value of a sorted dataset.

If $n$ is even:

$$
\text{Median} = \frac{x_{\frac{n}{2}} + x_{\frac{n}{2}+1}}{2}
$$

If $n$ is odd:

$$
\text{Median} = x_{\frac{n+1}{2}}
$$

The median is more robust to outliers compared to the mean.

### Mode

The mode is the most frequently occurring value in the dataset.

$$
\text{Mode} = \text{Value with highest frequency}
$$

It is especially useful for categorical or discrete numerical data.

## Dispersion Measures

### Variance (Sample Variance)

Variance measures the spread of data around the mean.

$$
s^2 = \frac{\sum (x_i - \bar{x})^2}{n - 1}
$$

A higher variance indicates greater dispersion.

### Standard Deviation

Standard deviation is the square root of variance.

$$
s = \sqrt{s^2}
$$

It is expressed in the same unit as the data, making it easier to interpret.


## Relationship Between Two Variables

### Covariance

Covariance measures how two variables vary together.

$$
\text{Cov}(X, Y) = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{n - 1}
$$

Interpretation:

* Positive value → variables move in same direction
* Negative value → variables move in opposite direction
* Near zero → weak linear relationship


### Correlation Coefficient (Pearson)

Correlation standardizes covariance:

$$
r = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y}
$$

Where:

* $\sigma_X$ = standard deviation of X
* $\sigma_Y$ = standard deviation of Y

The value of $r$ lies between -1 and +1.

* $+1$ → Perfect positive relationship
* $-1$ → Perfect negative relationship
* $0$ → No linear relationship


## Removing Redundant Feature

If:

$$
|r| > 0.8
$$

The features are considered highly correlated.

In such cases:

* One feature is removed
* Multicollinearity is reduced
* Model stability improves
* Redundant information is eliminated

Now this will render perfectly in Jupyter Notebook Markdown.


In [20]:
# Select column after outlier capping
col = 'chol'
data = list(new_df[col])
n = len(data)

# Calculate mean
total = 0
for value in data:
    total += value
mean = total / n
print(f"Mean: {mean:.2f}")

# Calculate median
sorted_data = sorted(data)
if n % 2 == 0:
    median = (sorted_data[n//2 - 1] + sorted_data[n//2]) / 2
else:
    median = sorted_data[n//2]
print(f"Median: {median:.2f}")

# Calculate mode
frequency = {}
for value in data:
    if value in frequency:
        frequency[value] += 1
    else:
        frequency[value] = 1
mode = max(frequency, key=frequency.get)
print(f"Mode: {mode}")

# Calculate variance
sum_squared_diff = 0
for value in data:
    sum_squared_diff += (value - mean) ** 2
variance = sum_squared_diff / (n - 1)
print(f"Variance: {variance:.2f}")

# Calculate standard deviation
std_dev = variance ** 0.5
print(f"Standard Deviation: {std_dev:.2f}")

# Select columns for covariance and correlation
column_1 = 'age'
column_2 = 'chol'

x = list(new_df[column_1])
y = list(new_df[column_2])
n = len(x)

# Calculate mean of both columns
mean_x = sum(x) / n
mean_y = sum(y) / n

# Calculate covariance
cov_sum = 0
for i in range(n):
    cov_sum += (x[i] - mean_x) * (y[i] - mean_y)
covariance = cov_sum / (n - 1)
print(f"Covariance: {covariance:.2f}")

# Calculate standard deviation of X
var_x = sum((value - mean_x) ** 2 for value in x) / (n - 1)
std_x = var_x ** 0.5

# Calculate standard deviation of Y
var_y = sum((value - mean_y) ** 2 for value in y) / (n - 1)
std_y = var_y ** 0.5

# Calculate correlation coefficient
if std_x != 0 and std_y != 0:
    correlation = covariance / (std_x * std_y)
else:
    correlation = 0

print(f"Correlation Coefficient (age vs chol): {correlation:.2f}")

# Remove redundant feature if highly correlated
if abs(correlation) > 0.8:
    print("Highly correlated. Removing one column.")
    new_df = new_df.drop(columns=[column_2])
else:
    print("Columns are not highly correlated.")

Mean: 245.38
Median: 240.50
Mode: 204.0
Variance: 2254.99
Standard Deviation: 47.49
Covariance: 85.46
Correlation Coefficient (age vs chol): 0.20
Columns are not highly correlated.


In [21]:
# Using Predefined functions
# Mean
mean = new_df[col].mean()
print(f"Mean: {mean:.2f}")

# Median
median = new_df[col].median()
print(f"Median: {median:.2f}")

# Mode
mode = new_df[col].mode()[0]
print(f"Mode: {mode}")

# Variance (sample variance)
variance = new_df[col].var()
print(f"Variance: {variance:.2f}")

# Standard Deviation
std_dev = new_df[col].std()
print(f"Standard Deviation: {std_dev:.2f}")

# Covariance
covariance = new_df[column_1].cov(new_df[column_2])
print(f"Covariance: {covariance:.2f}")

# Correlation
correlation = new_df[column_1].corr(new_df[column_2])
print(f"Correlation Coefficient (age vs chol): {correlation:.2f}")

# Multicollinearity check
if abs(correlation) > 0.8:
    print("Highly correlated. Removing one column.")
    new_df = new_df.drop(columns=[column_2])
else:
    print("Columns are not highly correlated.")

Mean: 245.38
Median: 240.50
Mode: 197.0
Variance: 2254.99
Standard Deviation: 47.49
Covariance: 85.46
Correlation Coefficient (age vs chol): 0.20
Columns are not highly correlated.


## Removing Highly Correlated Features

This step identifies and removes redundant numerical features based on a predefined correlation threshold.

### Process Overview

* A correlation threshold of **0.8** is defined.
* Only numerical columns are selected for correlation analysis.
* Columns with zero variance are removed since they contain no useful information.
* Columns with very low unique values (categorical-like behavior) are excluded to ensure meaningful correlation analysis.
* Pairwise correlation is calculated between all remaining numerical columns.
* If the absolute correlation between two features exceeds the threshold, one of them is marked for removal.
* Identified redundant columns are dropped from the dataset.

### Purpose

* Reduce multicollinearity
* Eliminate redundant information
* Improve model stability
* Reduce dimensionality
* Enhance interpretability

This ensures the dataset retains only meaningful and independent numerical features before further modeling or dimensionality reduction.


In [22]:
threshold = 0.8
columns_to_drop = set()

# Select numeric columns
numeric_columns = new_df.select_dtypes(include=[np.number]).columns

# Remove columns with zero variance
numeric_columns = [col for col in numeric_columns if new_df[col].std() != 0]

# Remove low unique columns (categorical-like)
numeric_columns = [col for col in numeric_columns if new_df[col].nunique() > 5]

for i in range(len(numeric_columns)):
    for j in range(i):
        col1 = numeric_columns[i]
        col2 = numeric_columns[j]

        correlation = new_df[col1].corr(new_df[col2])

        print(f"Checking {col1} vs {col2} → Correlation: {correlation:.2f}")

        if abs(correlation) > threshold:
            print(f"High correlation detected. Removing {col1}")
            columns_to_drop.add(col1)

new_df = new_df.drop(columns=columns_to_drop)

print("\nRemoved Columns:", list(columns_to_drop))
print("Remaining Columns:", list(new_df.columns))

Checking trestbps vs age → Correlation: 0.29
Checking chol vs age → Correlation: 0.20
Checking chol vs trestbps → Correlation: 0.14
Checking thalach vs age → Correlation: -0.39
Checking thalach vs trestbps → Correlation: -0.06
Checking thalach vs chol → Correlation: -0.01
Checking oldpeak vs age → Correlation: 0.21
Checking oldpeak vs trestbps → Correlation: 0.18
Checking oldpeak vs chol → Correlation: 0.05
Checking oldpeak vs thalach → Correlation: -0.35

Removed Columns: []
Remaining Columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


## Standardization (Z-Score Normalization)

Standardization is a feature scaling technique used to transform numerical variables so that they have:

* Mean equal to 0
* Standard deviation equal to 1

This process is also known as **Z-score normalization**.

## Z-Score Formula

$$
Z = \frac{X - \mu}{\sigma}
$$

Where:

* $X$ = original value
* $\mu$ = mean of the feature
* $\sigma$ = standard deviation of the feature

After transformation:

$$
\mu_Z = 0
$$

$$
\sigma_Z = 1
$$


## Purpose of Standardization

* Ensures all features are on the same scale
* Prevents large-scale features from dominating smaller-scale features
* Improves performance of distance-based algorithms
* Essential before applying PCA
* Helps gradient-based models converge faster


## Effect on Data

* The distribution shape remains the same
* Units are removed (dimensionless values)
* Relative distances between data points are preserved
* Outliers remain but are expressed in standard deviation units


## When to Use Standardization

Standardization is particularly important when:

* Features have different units (e.g., age vs cholesterol)
* Applying PCA
* Using algorithms like Logistic Regression, KNN, SVM, or Neural Networks

This step prepares the dataset for stable and fair model training.


In [23]:
# Apply Z-score normalization manually

for col in ['age','trestbps','chol','thalach','oldpeak']:
    mean = new_df[col].mean()
    std = new_df[col].std()
    
    new_df[col] = (new_df[col] - mean) / std

print("Z-score normalization applied.")

Z-score normalization applied.


In [25]:
# Dataset after Z-score standardization
new_df.head(10)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,-0.267522,1,0,-0.376886,-0.702872,0,1,0.807653,0,-0.025049,2,2.0,3.0,0
1,-0.157000,1,0,0.526444,-0.892399,0,0,0.236625,1,1.866169,0,0.0,3.0,0
2,1.721875,1,0,0.827554,-1.503097,0,1,-1.081130,1,1.415879,0,0.0,3.0,0
3,0.727176,1,0,1.008220,-0.892399,0,1,0.500176,0,-0.925629,2,1.0,3.0,0
4,0.837698,0,0,0.406000,1.023928,0,1,-1.915709,0,0.785473,1,2.5,2.0,0
5,0.395610,0,0,-1.882435,0.055235,0,0,-1.212906,0,-0.025049,1,0.0,2.0,1
6,0.395610,1,0,-1.039328,1.529333,0,2,-0.422253,0,2.676691,0,2.5,1.0,0
7,0.064044,1,0,1.730883,0.918635,0,0,-0.202627,1,-0.205165,1,1.0,3.0,0
8,-0.930654,1,0,-0.677996,0.076294,0,0,-0.246552,0,-0.205165,2,0.0,3.0,0
9,-0.046478,1,0,-0.557552,0.855459,0,0,-1.476457,1,1.956227,1,2.0,2.0,0


## Normalization (Min–Max Scaling)

Normalization is a feature scaling technique that rescales numerical values into a fixed range, typically between 0 and 1.

This method preserves the relative relationships between data points while changing their scale.


## Min–Max Normalization Formula

$$
X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}
$$

Where:

* $X$ = original value
* $X_{min}$ = minimum value of the feature
* $X_{max}$ = maximum value of the feature

After transformation:

$$
0 \leq X_{norm} \leq 1
$$

## Purpose of Normalization

* Brings all features into the same bounded range
* Prevents dominance of large-value features
* Improves performance of distance-based models
* Useful for neural networks and gradient-based algorithms


## Effect on Data

* Distribution shape remains the same
* Values are scaled proportionally
* Minimum value becomes 0
* Maximum value becomes 1
* Sensitive to outliers (extreme values affect scaling range)


## When to Use Normalization

Normalization is preferred when:

* Features have different units and scales
* Data must be bounded between 0 and 1
* Using algorithms like KNN, K-Means, or Neural Networks

It is important to perform outlier handling before normalization because extreme values can significantly distort the scaling range.


In [26]:
# Or you can apply Min-Max normalization manually as well 

for col in ['age','trestbps','chol','thalach','oldpeak']:
    min_val = new_df[col].min()
    max_val = new_df[col].max()
    
    new_df[col] = (new_df[col] - min_val) / (max_val - min_val)

print("Min-Max normalization applied.")